# AnchorKV: complete T4 research notebook

Upload this one notebook to a **fresh Colab T4 runtime**, then run all cells.
No GitHub checkout, previous notebook, HF token, or results zip is required.
It includes: validated chat prompts and answers; equal-byte-budget selectors;
automatic query/error scoring and a quantization-sensitivity audit; continuously
demoted KV pages; experimental packed Triton attention; repeated measurements;
and a downloadable report with machine-readable results and source snapshots.

Default pilot: six prompts, two context lengths, three evidence positions,
three timing repetitions. Allow tens of minutes; Python page allocation and
reference fallbacks can be slow. A larger suite is configurable below.
The automatic score is a new hypothesis, not an already validated thought-anchor
detector. The oracle uses labeled evidence solely as a comparison. Packed
attention must pass numerical tests on your GPU before its results are accepted.

Scope: Qwen3-0.6B, FP16 weights, batch one, full attention, single-token decode.
This is an experimental kernel and benchmark, not a vLLM integration or a full
reproduction of model-generated Declarative Attention tags.


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.57.6', 'huggingface_hub>=0.34,<1',
                       'accelerate>=1,<2', 'pandas', 'matplotlib'])


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import random
import shutil
import statistics
import sys
import time
import traceback

import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers

assert transformers.__version__ == '4.57.6', 'Restart the Colab session after installing, then run from the top.'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__, '| Transformers:', transformers.__version__)


In [ ]:
# Embedded runtime: generated from the repository, no network checkout required.
EMBEDDED_SOURCES = {'__init__.py': '', 'packed_decode.py': '"""Experimental append-only, batch-one KV pages for the standalone Colab suite.\n\nPayloads remain on their input device. Demoted pages release their FP16 owners;\nthe pointer table addresses independent allocations rather than a padded FP16\narena. This implementation favors inspectability over allocation throughput.\n"""\n\nfrom dataclasses import dataclass\nimport math\nimport random\n\nimport torch\nimport torch.nn.functional as F\n\n\ndef tensor_bytes(tensor):\n    return tensor.numel() * tensor.element_size()\n\n\n@dataclass\nclass QuantizedPage:\n    bits: int\n    data: torch.Tensor\n    scales: torch.Tensor | None\n    shape: tuple\n    group: int\n\n    @property\n    def nbytes(self):\n        return tensor_bytes(self.data) + (\n            tensor_bytes(self.scales) if self.scales is not None else 0\n        )\n\n    def dense(self):\n        if self.bits == 16:\n            return self.data\n        if self.bits == 8:\n            integers = self.data.float()\n        else:\n            lo = self.data & 15\n            hi = self.data >> 4\n            values = torch.stack((lo, hi), -1).reshape(self.shape).short()\n            integers = torch.where(values >= 8, values - 16, values).float()\n        groups = integers.reshape(*self.shape[:-1], -1, self.group)\n        return (groups * self.scales.float().unsqueeze(-1)).reshape(self.shape).half()\n\n\ndef pack_page(tensor, bits=4, group=64):\n    if bits not in (4, 8, 16):\n        raise ValueError(\'bits must be 4, 8, or 16\')\n    if tensor.ndim != 3 or tensor.shape[-1] % group or group % 2:\n        raise ValueError(\'expected [KV heads, page tokens, head dim], with even groups\')\n    source = tensor.detach().half().contiguous()\n    if bits == 16:\n        # Own storage: do not retain a view of the original full prompt cache.\n        return QuantizedPage(bits, source.clone(), None, tuple(source.shape), group)\n    groups = source.float().reshape(*source.shape[:-1], -1, group)\n    maxima = groups.abs().amax(-1)\n    qmax = 7 if bits == 4 else 127\n    # Clamp before FP16 conversion: tiny nonzero groups must not get scale zero.\n    scales = (maxima / qmax).clamp_min(torch.finfo(torch.float16).tiny).half()\n    integers = torch.round(groups / scales.float().unsqueeze(-1))\n    integers = integers.clamp(-qmax, qmax).to(torch.int8).reshape(source.shape)\n    if bits == 4:\n        nibble = (integers.short() & 15).byte()\n        integers = (nibble[..., 0::2] | (nibble[..., 1::2] << 4)).contiguous()\n    return QuantizedPage(bits, integers, scales, tuple(source.shape), group)\n\n\ndef choose_pages(name, candidates, count, *, scores=None, evidence=(), seed=0):\n    """Choose exactly the same number of full pages for every mixed policy."""\n    candidates = sorted(set(candidates))\n    if not 0 <= count <= len(candidates):\n        raise ValueError(\'invalid high-precision page allowance\')\n    if name == \'random\':\n        order = random.Random(seed).sample(candidates, len(candidates))\n    elif name == \'recent\':\n        order = sorted(candidates, reverse=True)\n    elif name == \'automatic\':\n        if scores is None or any(page not in scores for page in candidates):\n            raise ValueError(\'automatic selection requires every candidate score\')\n        order = sorted(candidates, key=lambda page: (-scores[page], page))\n    elif name == \'oracle\':\n        evidence = set(evidence)\n        order = sorted(candidates, key=lambda page: (page not in evidence, -page))\n    else:\n        raise ValueError(f\'unknown selector: {name}\')\n    return frozenset(order[:count])\n\n\ndef rank_correlation(x, y):\n    def ranks(values):\n        order = sorted(range(len(values)), key=lambda i: values[i])\n        out = torch.empty(len(values), dtype=torch.float64)\n        start = 0\n        while start < len(order):\n            end = start + 1\n            while end < len(order) and values[order[end]] == values[order[start]]:\n                end += 1\n            out[order[start:end]] = (start + end - 1) / 2\n            start = end\n        return out - out.mean()\n    a, b = ranks(x), ranks(y)\n    denominator = a.norm() * b.norm()\n    return float(a.dot(b) / denominator) if denominator > 0 else None\n\n\nclass PagedLayer:\n    def __init__(self, heads, dim, max_tokens, *, block=16, group=64,\n                 archive_bits=4, protected=(), recent_pages=2, device=\'cuda\'):\n        if dim % group or archive_bits not in (4, 8, 16):\n            raise ValueError(\'invalid quantization geometry\')\n        self.heads, self.dim, self.block, self.group = heads, dim, block, group\n        self.archive_bits = archive_bits\n        self.protected = frozenset(protected)\n        self.recent_pages = recent_pages\n        self.capacity = math.ceil(max_tokens / block)\n        self.device = torch.device(device)\n        self.pages = []\n        self.length = 0\n        self.demotions = 0\n        self.table = torch.zeros((self.capacity, 5), dtype=torch.int64, device=device)\n        self.dummy_scale = torch.ones(1, dtype=torch.float16, device=device)\n        self.archived_through = -1\n\n    def _publish(self, page):\n        key, value = self.pages[page]\n        ks = self.dummy_scale if key.scales is None else key.scales\n        vs = self.dummy_scale if value.scales is None else value.scales\n        self.table[page] = torch.tensor(\n            [key.data.data_ptr(), value.data.data_ptr(), ks.data_ptr(), vs.data_ptr(), key.bits],\n            dtype=torch.int64, device=self.device,\n        )\n\n    def append(self, key, value):\n        if key.shape != value.shape or key.ndim != 4 or key.shape[:2] != (1, self.heads):\n            raise ValueError(\'only batch-one matching K/V tensors are supported\')\n        if key.shape[-1] != self.dim or self.length + key.shape[-2] > self.capacity * self.block:\n            raise ValueError(\'cache geometry or capacity exceeded\')\n        offset = 0\n        while offset < key.shape[-2]:\n            page, position = divmod(self.length, self.block)\n            if position == 0:\n                empty = torch.zeros((self.heads, self.block, self.dim),\n                                    dtype=torch.float16, device=self.device)\n                self.pages.append((pack_page(empty, 16, self.group), pack_page(empty, 16, self.group)))\n                self._publish(page)\n            take = min(self.block - position, key.shape[-2] - offset)\n            kpage, vpage = self.pages[page]\n            kpage.data[:, position:position + take].copy_(key[0, :, offset:offset + take])\n            vpage.data[:, position:position + take].copy_(value[0, :, offset:offset + take])\n            self.length += take\n            offset += take\n            self._archive_old_pages()\n\n    def _archive_old_pages(self):\n        # Complete pages only; the partial append page always stays FP16.\n        last_eligible = self.length // self.block - self.recent_pages - 1\n        for page in range(self.archived_through + 1, last_eligible + 1):\n            if page not in self.protected and self.archive_bits != 16:\n                self.demote(page, self.archive_bits)\n        self.archived_through = max(self.archived_through, last_eligible)\n\n    def demote(self, page, bits):\n        key, value = self.pages[page]\n        if bits > key.bits:\n            raise ValueError(\'lossy promotion cannot restore FP16 values\')\n        if page in self.protected and bits != 16:\n            raise ValueError(\'cannot demote a protected page\')\n        if bits != key.bits:\n            self.pages[page] = (pack_page(key.dense(), bits, self.group),\n                                pack_page(value.dense(), bits, self.group))\n            self._publish(page)\n            self.demotions += 1\n\n    def protect(self, page):\n        if page < len(self.pages) and self.pages[page][0].bits != 16:\n            raise ValueError(\'anchor must be declared before demotion\')\n        self.protected = self.protected | {page}\n\n    @property\n    def payload_bytes(self):\n        return sum(k.nbytes + v.nbytes for k, v in self.pages)\n\n    @property\n    def resident_bytes(self):\n        return self.payload_bytes + tensor_bytes(self.table) + tensor_bytes(self.dummy_scale)\n\n    def dense(self):\n        if not self.pages:\n            raise ValueError(\'cache is empty\')\n        k = torch.cat([pair[0].dense() for pair in self.pages], dim=1)[:, :self.length]\n        v = torch.cat([pair[1].dense() for pair in self.pages], dim=1)[:, :self.length]\n        return k.unsqueeze(0), v.unsqueeze(0)\n\n\ndef dense_attention(query, layer, scale):\n    key, value = layer.dense()\n    repeats = query.shape[1] // key.shape[1]\n    return F.scaled_dot_product_attention(\n        query, key.repeat_interleave(repeats, 1), value.repeat_interleave(repeats, 1),\n        is_causal=False, scale=scale,\n    )\n\n\ndef kl_divergence(reference, candidate):\n    p = reference.float().log_softmax(-1)\n    q = candidate.float().log_softmax(-1)\n    return (p.exp() * (p - q)).sum(-1).clamp_min(0)\n', 'triton_decode.py': '"""Experimental mixed-page single-query GQA attention, FP16/INT8/packed INT4.\n\nEach split streams pages, unpacks only the current tile in registers, and\nmaintains online softmax statistics. A second kernel merges split statistics.\nNo sequence-length dense K/V tensor is created. GPU gates in the notebook\nmust pass before this backend contributes benchmark results.\n"""\n\nimport torch\nimport triton\nimport triton.language as tl\n\n\n@triton.jit\ndef _page_values(address, scale_address, mode, head, rows, cols,\n                 BLOCK: tl.constexpr, DIM: tl.constexpr, GROUP: tl.constexpr):\n    offsets = (head * BLOCK + rows[:, None]) * DIM + cols[None, :]\n    if mode == 16:\n        values = tl.load(address.to(tl.pointer_type(tl.float16)) + offsets)\n    else:\n        if mode == 8:\n            integers = tl.load(address.to(tl.pointer_type(tl.int8)) + offsets).to(tl.float32)\n        else:\n            packed_offsets = (head * BLOCK + rows[:, None]) * (DIM // 2) + cols[None, :] // 2\n            packed = tl.load(address.to(tl.pointer_type(tl.uint8)) + packed_offsets).to(tl.int32)\n            nibble = (packed >> ((cols[None, :] % 2) * 4)) & 15\n            integers = tl.where(nibble >= 8, nibble - 16, nibble).to(tl.float32)\n        scale_offsets = (head * BLOCK + rows[:, None]) * (DIM // GROUP) + cols[None, :] // GROUP\n        scales = tl.load(scale_address.to(tl.pointer_type(tl.float16)) + scale_offsets)\n        # Match the FP16 execution values of the dense reference exactly.\n        values = (integers * scales.to(tl.float32)).to(tl.float16)\n    return values.to(tl.float32)\n\n\n@triton.jit\ndef _decode_parts(Q, TABLE, PART, n_tokens, scale,\n                  DIM: tl.constexpr, BLOCK: tl.constexpr,\n                  GROUP: tl.constexpr, GQA: tl.constexpr, SPLITS: tl.constexpr):\n    head = tl.program_id(0)\n    split = tl.program_id(1)\n    kv_head = head // GQA\n    cols = tl.arange(0, DIM)\n    rows = tl.arange(0, BLOCK)\n    query = tl.load(Q + head * DIM + cols).to(tl.float32)\n    maximum = tl.full((), float(\'-inf\'), tl.float32)\n    denominator = tl.full((), 0.0, tl.float32)\n    numerator = tl.full((DIM,), 0.0, tl.float32)\n    for page in range(split, tl.cdiv(n_tokens, BLOCK), SPLITS):\n        kptr = tl.load(TABLE + page * 5)\n        vptr = tl.load(TABLE + page * 5 + 1)\n        kscale = tl.load(TABLE + page * 5 + 2)\n        vscale = tl.load(TABLE + page * 5 + 3)\n        mode = tl.load(TABLE + page * 5 + 4)\n        keys = _page_values(kptr, kscale, mode, kv_head, rows, cols, BLOCK, DIM, GROUP)\n        values = _page_values(vptr, vscale, mode, kv_head, rows, cols, BLOCK, DIM, GROUP)\n        scores = tl.sum(keys * query[None, :], 1) * scale\n        scores = tl.where(page * BLOCK + rows < n_tokens, scores, float(\'-inf\'))\n        new_maximum = tl.maximum(maximum, tl.max(scores, 0))\n        rescale = tl.exp(maximum - new_maximum)\n        probabilities = tl.exp(scores - new_maximum)\n        numerator = numerator * rescale + tl.sum(probabilities[:, None] * values, 0)\n        denominator = denominator * rescale + tl.sum(probabilities, 0)\n        maximum = new_maximum\n    base = (head * SPLITS + split) * (DIM + 2)\n    tl.store(PART + base + cols, numerator)\n    tl.store(PART + base + DIM, maximum)\n    tl.store(PART + base + DIM + 1, denominator)\n\n\n@triton.jit\ndef _merge_parts(PART, OUT, DIM: tl.constexpr, SPLITS: tl.constexpr):\n    head = tl.program_id(0)\n    splits = tl.arange(0, SPLITS)\n    cols = tl.arange(0, DIM)\n    base = (head * SPLITS + splits) * (DIM + 2)\n    maxima = tl.load(PART + base + DIM)\n    denominators = tl.load(PART + base + DIM + 1)\n    weights = tl.exp(maxima - tl.max(maxima, 0))\n    numerators = tl.load(PART + base[:, None] + cols[None, :])\n    denominator = tl.sum(weights * denominators, 0)\n    output = tl.sum(numerators * weights[:, None], 0) / denominator\n    tl.store(OUT + head * DIM + cols, output)\n\n\ndef packed_attention(query, layer, scale, splits=4):\n    if query.shape[0] != 1 or query.shape[2] != 1 or query.shape[-1] != layer.dim:\n        raise ValueError(\'packed kernel supports batch=1, query length=1 only\')\n    if query.dtype != torch.float16 or not query.is_cuda or layer.length == 0:\n        raise ValueError(\'packed kernel requires a nonempty CUDA FP16 cache\')\n    if layer.dim not in (64, 128) or layer.block != 16 or layer.group != 64:\n        raise ValueError(\'supported geometry: block=16, group=64, head dim=64/128\')\n    heads = query.shape[1]\n    if heads % layer.heads:\n        raise ValueError(\'query heads must be divisible by KV heads\')\n    query = query.contiguous()\n    partial = torch.empty((heads, splits, layer.dim + 2), device=query.device, dtype=torch.float32)\n    output = torch.empty_like(query)\n    _decode_parts[(heads, splits)](\n        query, layer.table, partial, layer.length, scale,\n        DIM=layer.dim, BLOCK=layer.block, GROUP=layer.group,\n        GQA=heads // layer.heads, SPLITS=splits, num_warps=4,\n    )\n    _merge_parts[(heads,)](partial, output, DIM=layer.dim, SPLITS=splits, num_warps=4)\n    return output\n', 'colab_experiment.py': '"""Reproducible experiment helpers embedded in the all-in-one Colab notebook."""\n\nfrom contextlib import contextmanager\nfrom dataclasses import asdict, dataclass\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport re\nimport statistics\nimport time\n\nimport torch\n\nfrom .packed_decode import PagedLayer, choose_pages, dense_attention, kl_divergence, pack_page\n\n\n@dataclass\nclass Settings:\n    model_id: str = \'Qwen/Qwen3-0.6B\'\n    revision: str = \'c1899de289a04d12100db370d81485cdf75e47ca\'\n    seed: int = 7\n    block: int = 16\n    recent_pages: int = 2\n    keep_fraction: float = 0.15\n    max_new_tokens: int = 48\n    max_prompt_tokens: int = 1536\n    repeats: int = 3\n    score_layers: int = 4\n    sensitivity_pages: int = 8\n    lengths: tuple = (384, 896)\n    positions: tuple = (\'early\', \'middle\', \'late\')\n\n\ndef sync():\n    torch.cuda.synchronize()\n\n\ndef clean():\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\ndef build_case(tokenizer, settings, index, target_length, position):\n    rng = random.Random(settings.seed + index)\n    answer = str(rng.randrange(1000, 9999))\n    project = [\'Zephyr\', \'Juniper\', \'Cobalt\', \'Orion\', \'Maple\', \'Atlas\'][index % 6]\n    evidence = f\'Authoritative record: the access code for Project {project} is {answer}.\'\n    instruction = \'Read the records and return only the requested four-digit code. Do not explain or repeat it.\'\n    query = f\'What is the access code for Project {project}?\'\n    def render(repetitions):\n        decoys = [f\'Retired record {i}: Project Cedar{i} used code {rng.randrange(1000, 9999)}. \'\n                  \'This expired record is unrelated to the requested project.\' for i in range(repetitions)]\n        location = {\'early\': 0, \'middle\': len(decoys) // 2, \'late\': len(decoys)}[position]\n        decoys.insert(location, evidence)\n        text = \'\\n\'.join(decoys) + \'\\n\\n\' + query\n        return tokenizer.apply_chat_template(\n            [{\'role\': \'system\', \'content\': instruction}, {\'role\': \'user\', \'content\': text}],\n            tokenize=False, add_generation_prompt=True, enable_thinking=False,\n        )\n    # Fit complete records before loading weights; never truncate the query/answer marker.\n    best = None\n    for repetitions in range(1, 65):\n        text = render(repetitions)\n        encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)\n        if len(encoded.input_ids) > min(target_length, settings.max_prompt_tokens):\n            break\n        best = text, encoded\n    if best is None:\n        raise ValueError(\'context bound is too small for the chat scaffold and evidence\')\n    text, encoded = best\n    start = text.index(evidence)\n    end = start + len(evidence)\n    evidence_tokens = [i for i, (a, b) in enumerate(encoded.offset_mapping) if b > start and a < end]\n    return {\'case_id\': f\'case-{index:03}\', \'answer\': answer, \'position\': position,\n            \'target_length\': target_length, \'prompt\': text, \'ids\': encoded.input_ids,\n            \'evidence_pages\': sorted({i // settings.block for i in evidence_tokens}),\n            \'prompt_sha256\': hashlib.sha256(text.encode()).hexdigest()}\n\n\ndef stock_cache(source, device=\'cuda\'):\n    from transformers import DynamicCache\n    return DynamicCache([(k.to(device), v.to(device)) for k, v in source])\n\n\ndef cache_cpu(cache):\n    return [(layer.keys.detach().cpu(), layer.values.detach().cpu()) for layer in cache.layers]\n\n\ndef make_live_cache(source, settings, protected=(), bits=4, max_tokens=None, device=\'cuda\'):\n    from transformers import DynamicCache\n    class LiveCache(DynamicCache):\n        def __init__(self):\n            super().__init__()\n            self.stores = []\n            for key, value in source:\n                store = PagedLayer(key.shape[1], key.shape[-1], max_tokens,\n                                   block=settings.block, archive_bits=bits,\n                                   protected=protected, recent_pages=settings.recent_pages, device=device)\n                store.append(key.to(device), value.to(device))\n                self.stores.append(store)\n\n        def get_seq_length(self, layer_idx=0):\n            return self.stores[layer_idx].length\n\n        def update(self, key_states, value_states, layer_idx, cache_kwargs=None):\n            self.stores[layer_idx].append(key_states, value_states)\n            # These are ignored by our attention interface, which reads the stores.\n            return key_states, value_states\n\n        def report(self):\n            return {\'resident_bytes\': sum(s.resident_bytes for s in self.stores),\n                    \'payload_bytes\': sum(s.payload_bytes for s in self.stores),\n                    \'demotions\': sum(s.demotions for s in self.stores),\n                    \'tokens\': self.get_seq_length()}\n    if max_tokens is None:\n        max_tokens = source[0][0].shape[-2] + settings.max_new_tokens + 32\n    return LiveCache()\n\n\n@contextmanager\ndef attention_route(model, cache=None, backend=\'dense\', capture=None):\n    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS\n    original = model.config._attn_implementation\n    dense_interface = ALL_ATTENTION_FUNCTIONS[\'sdpa\']\n    def attention(module, query, key, value, attention_mask, scaling, **kwargs):\n        if capture is not None:\n            capture[module.layer_idx] = query.detach().cpu()\n        if cache is None:\n            return dense_interface(module, query, key, value, attention_mask,\n                                   scaling=scaling, **kwargs)\n        store = cache.stores[module.layer_idx]\n        if backend == \'packed\':\n            from .triton_decode import packed_attention\n            result = packed_attention(query, store, scaling)\n        else:\n            result = dense_attention(query, store, scaling)\n        return result.transpose(1, 2).contiguous(), None\n    ALL_ATTENTION_FUNCTIONS.register(\'anchorkv_experiment\', attention)\n    model.config._attn_implementation = \'anchorkv_experiment\'\n    try:\n        yield\n    finally:\n        model.config._attn_implementation = original\n        # The global registry must not retain a closure owning the previous GPU cache.\n        ALL_ATTENTION_FUNCTIONS.register(\'anchorkv_experiment\', dense_interface)\n\n\ndef forward_token(model, cache, token, packed=False):\n    position = cache.get_seq_length()\n    return model(input_ids=torch.tensor([[token]], device=model.device), past_key_values=cache,\n                 cache_position=torch.tensor([position], device=model.device),\n                 attention_mask={\'full_attention\': None} if packed else None,\n                 use_cache=True, logits_to_keep=1)\n\n\n@torch.inference_mode()\ndef prefill_case(model, case):\n    # Leave the final prompt token out so every policy computes its own FIRST answer logit.\n    ids = torch.tensor([case[\'ids\'][:-1]], device=\'cuda\')\n    output = model(input_ids=ids, use_cache=True, logits_to_keep=1)\n    source = cache_cpu(output.past_key_values)\n    del output, ids\n    clean()\n    return source\n\n\n@torch.inference_mode()\ndef automatic_scores(model, source, case, candidates, settings):\n    started = time.perf_counter()\n    captured = {}\n    cache = stock_cache(source)\n    with attention_route(model, capture=captured):\n        output = forward_token(model, cache, case[\'ids\'][-1])\n    reference_logits = output.logits[0, -1].float().cpu()\n    del cache, output\n    # Query tensors from the last prompt token only. No gold answer/evidence labels enter this score.\n    scores = {page: 0.0 for page in candidates}\n    layers = sorted(captured)[-settings.score_layers:]\n    for idx in layers:\n        q = captured[idx].float()\n        key, value = source[idx]\n        repeats = q.shape[1] // key.shape[1]\n        probabilities = (q @ key.float().repeat_interleave(repeats, 1).transpose(-2, -1))\n        probabilities = (probabilities / math.sqrt(key.shape[-1])).softmax(-1).mean((0, 1, 2))\n        for page in candidates:\n            start, end = page * settings.block, (page + 1) * settings.block\n            k = key[0, :, start:end].contiguous()\n            v = value[0, :, start:end].contiguous()\n            k_error = (pack_page(k, 4).dense().float() - k.float()).square().mean()\n            v_error = (pack_page(v, 4).dense().float() - v.float()).square().mean()\n            scores[page] += float(probabilities[start:end].sum() * (k_error + v_error)) / len(layers)\n    sync()\n    return scores, reference_logits, time.perf_counter() - started\n\n\ndef eos_ids(model, tokenizer):\n    value = model.generation_config.eos_token_id\n    ids = set(value if isinstance(value, list) else [value])\n    ids.add(tokenizer.eos_token_id)\n    return ids - {None}\n\n\ndef answer_check(text, answer, ended):\n    return {\'answer_correct\': text.strip() == answer,\n            \'completed_correct\': ended and text.strip() == answer,\n            \'ended_eos\': ended, \'truncated\': not ended}\n\n\n@torch.inference_mode()\ndef run_policy(model, tokenizer, source, case, settings, policy, protected=(),\n               backend=\'packed\', forced=None):\n    clean()\n    sync()\n    torch.cuda.reset_peak_memory_stats()\n    base_alloc = torch.cuda.memory_allocated()\n    started = time.perf_counter()\n    native = policy == \'native_fp16\'\n    bits = 16 if policy == \'paged_fp16\' else 8 if policy == \'int8\' else 4\n    cache = stock_cache(source) if native else make_live_cache(source, settings, protected, bits)\n    sync()\n    setup_seconds = time.perf_counter() - started\n    payload_before = cache.report() if not native else {\'tokens\': cache.get_seq_length()}\n    decode_started = time.perf_counter()\n    outputs, logits = [], []\n    token = case[\'ids\'][-1]\n    stopped = False\n    steps = len(forced) if forced is not None else settings.max_new_tokens\n    route_cache = None if native else cache\n    with attention_route(model, cache=route_cache, backend=backend):\n        for step in range(steps):\n            output = forward_token(model, cache, token, packed=not native)\n            current = output.logits[0, -1].float()\n            logits.append(current.detach().cpu())\n            # Gold tokens condition the diagnostic replay only, never the free generation.\n            token = int(forced[step]) if forced is not None else int(current.argmax())\n            outputs.append(token)\n            if forced is None and token in eos_ids(model, tokenizer):\n                stopped = True\n                break\n    sync()\n    decode_seconds = time.perf_counter() - decode_started\n    # Starts before cache transfer, packing, table creation, materialization and generation.\n    total_seconds = time.perf_counter() - started\n    peak = torch.cuda.max_memory_allocated()\n    live_alloc = torch.cuda.memory_allocated()\n    if native:\n        physical = sum(t.numel() * t.element_size() for layer in cache.layers for t in (layer.keys, layer.values))\n        report = {\'resident_bytes\': physical, \'payload_bytes\': physical,\n                  \'demotions\': 0, \'tokens\': cache.get_seq_length()}\n    else:\n        report = cache.report()\n    text = tokenizer.decode(outputs, skip_special_tokens=True)\n    row = {\'policy\': policy, \'backend\': \'stock_sdpa\' if native else backend,\n           \'case_id\': case[\'case_id\'], \'text\': text, \'generated_tokens\': len(outputs),\n           **answer_check(text, case[\'answer\'], stopped), **report,\n           \'initial_resident_bytes\': payload_before.get(\'resident_bytes\'),\n           \'setup_seconds\': setup_seconds, \'decode_seconds\': decode_seconds,\n           \'decode_tokens_per_second\': len(outputs) / decode_seconds,\n           \'cache_plus_decode_seconds\': total_seconds,\n           \'peak_allocated_gib\': peak / 2**30,\n           \'peak_incremental_mib\': (peak - base_alloc) / 2**20,\n           \'end_incremental_mib\': (live_alloc - base_alloc) / 2**20}\n    del cache, output, current\n    clean()\n    return row, torch.stack(logits)\n\n\n@torch.inference_mode()\ndef kernel_gate():\n    from .triton_decode import packed_attention\n    checks = []\n    for dim in (64, 128):\n        for length in (1, 15, 16, 17, 33, 65, 97):\n            for bits in (4, 8, 16):\n                layer = PagedLayer(2, dim, 128, archive_bits=bits, protected=(0,), recent_pages=1)\n                k = torch.randn(1, 2, length, dim, device=\'cuda\', dtype=torch.float16)\n                v = torch.randn_like(k)\n                # Append in uneven increments to exercise partial blocks and online demotion.\n                for begin in range(0, length, 7):\n                    layer.append(k[:, :, begin:begin + 7], v[:, :, begin:begin + 7])\n                q = torch.randn(1, 4, 1, dim, device=\'cuda\', dtype=torch.float16)\n                reference = dense_attention(q, layer, dim ** -0.5)\n                actual = packed_attention(q, layer, dim ** -0.5)\n                torch.testing.assert_close(actual, reference, atol=0.003, rtol=0.003)\n                checks.append({\'dim\': dim, \'tokens\': length, \'bits\': bits,\n                               \'max_error\': float((actual - reference).abs().max()),\n                               \'demotions\': layer.demotions})\n    return checks\n\n\n@torch.inference_mode()\ndef model_gate(model, tokenizer, source, case, settings):\n    forced = tokenizer(case[\'answer\'], add_special_tokens=False).input_ids\n    _, stock = run_policy(model, tokenizer, source, case, settings, \'native_fp16\', forced=forced)\n    checks = []\n    for policy, protected in [(\'paged_fp16\', ()), (\'int8\', ()), (\'int4\', ()), (\'oracle\', (0, 2))]:\n        _, dense = run_policy(model, tokenizer, source, case, settings, policy,\n                              protected, backend=\'dense\', forced=forced)\n        _, packed = run_policy(model, tokenizer, source, case, settings, policy,\n                               protected, backend=\'packed\', forced=forced)\n        discrepancy = float(kl_divergence(dense, packed).mean())\n        if discrepancy > 0.002 or not torch.isfinite(packed).all():\n            raise AssertionError(f\'kernel/model mismatch for {policy}: KL={discrepancy}\')\n        if policy == \'paged_fp16\':\n            torch.testing.assert_close(dense, stock, atol=0.05, rtol=0.01)\n            if float(kl_divergence(stock, packed).mean()) > 0.002:\n                raise AssertionError(\'FP16 packed route does not match the stock model\')\n        checks.append({\'policy\': policy, \'mean_kl_dense_vs_packed\': discrepancy,\n                       \'max_logit_error\': float((dense - packed).abs().max())})\n    return checks\n\n\n@torch.inference_mode()\ndef sensitivity_audit(model, source, case, settings, scores, reference):\n    """Diagnostic only: never feed these measured labels back into the selector."""\n    from .packed_decode import rank_correlation\n    candidates = sorted(scores)\n    if len(candidates) > settings.sensitivity_pages:\n        indices = torch.linspace(0, len(candidates) - 1, settings.sensitivity_pages).round().int().tolist()\n        candidates = [candidates[i] for i in indices]\n    rows = []\n    for page in candidates:\n        cache = make_live_cache(source, settings, bits=16)\n        for layer in cache.stores:\n            layer.demote(page, 4)\n        with attention_route(model, cache, backend=\'dense\'):\n            output = forward_token(model, cache, case[\'ids\'][-1], packed=True)\n        measured = float(kl_divergence(reference, output.logits[0, -1].float().cpu()))\n        rows.append({\'page\': page, \'automatic_score\': scores[page], \'measured_kl\': measured})\n        del cache, output\n    correlation = rank_correlation([r[\'automatic_score\'] for r in rows], [r[\'measured_kl\'] for r in rows])\n    return {\'case_id\': case[\'case_id\'], \'spearman\': correlation, \'pages\': rows,\n            \'scope\': \'single-page INT4 intervention; first answer-token distribution only\'}\n\n\ndef bootstrap_interval(values, seed=7, iterations=2000):\n    if not values:\n        return [None, None]\n    rng = random.Random(seed)\n    means = sorted(statistics.mean(rng.choices(values, k=len(values))) for _ in range(iterations))\n    return [means[int(iterations * 0.025)], means[int(iterations * 0.975)]]\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n'}
EMBEDDED_SOURCE_SHA256 = '9528b35c89eff4b2c4c13f39c29d2a3224b5f9d944e88c4e0feaeb4f6db78b30'
runtime_root = Path('/content/anchorkv-embedded-runtime')
runtime_package = runtime_root / 'anchorkv_notebook'
runtime_package.mkdir(parents=True, exist_ok=True)
for filename, source in EMBEDDED_SOURCES.items():
    (runtime_package / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, str(runtime_root))


In [ ]:
from anchorkv_notebook.colab_experiment import (
    Settings, asdict, build_case, prefill_case, automatic_scores, choose_pages,
    run_policy, kernel_gate, model_gate, sensitivity_audit, bootstrap_interval,
    save_json, clean, sync, eos_ids,
)
from anchorkv_notebook.packed_decode import kl_divergence
from transformers import AutoTokenizer, AutoModelForCausalLM

settings = Settings()
# Optional expanded run (slower):
# settings.lengths = (384, 896, 1408)
# settings.repeats = 5
# settings.sensitivity_pages = 16
# Re-run with different settings.seed values for broader evidence.
REQUEST_PACKED_KERNEL = True
RUN_SENSITIVITY_AUDIT = True
OUTPUT = Path('/content/anchorkv-complete-results') / time.strftime('%Y%m%d-%H%M%S')
OUTPUT.mkdir(parents=True, exist_ok=True)
torch.manual_seed(settings.seed)
torch.cuda.manual_seed_all(settings.seed)
random.seed(settings.seed)
torch.set_num_threads(2)

tokenizer = AutoTokenizer.from_pretrained(settings.model_id, revision=settings.revision)
cases = []
for target in settings.lengths:
    for position in settings.positions:
        cases.append(build_case(tokenizer, settings, len(cases), target, position))
assert all(len(case['ids']) <= settings.max_prompt_tokens for case in cases)
display(pd.DataFrame([{'case': c['case_id'], 'tokens': len(c['ids']),
                      'evidence_position': c['position'], 'answer': c['answer']} for c in cases]))
save_json(OUTPUT / 'prompts.json', cases)
save_json(OUTPUT / 'settings.json', asdict(settings))


## Load weights after validating every prompt

Prompt length is checked before GPU model allocation. Each prompt uses Qwen's
chat template with thinking disabled for the retrieval benchmark. Generated
answers must equal the four-digit code; EOS completion is scored separately.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    settings.model_id, revision=settings.revision, torch_dtype=torch.float16,
    attn_implementation='sdpa', low_cpu_mem_usage=True,
).to('cuda').eval()
assert not getattr(model.model, 'has_sliding_layers', False), 'Only full-attention Qwen3 is supported.'
assert model.config.head_dim == 128
try:
    triton_version = importlib.metadata.version('triton')
except importlib.metadata.PackageNotFoundError:
    triton_version = None
environment = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'transformers': transformers.__version__, 'triton': triton_version,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
    'compute_capability': list(torch.cuda.get_device_capability(0)),
    'model_id': settings.model_id, 'model_revision': settings.revision,
    'source_sha256': EMBEDDED_SOURCE_SHA256,
}
save_json(OUTPUT / 'environment.json', environment)
print(environment)


## Numerical gates for the packed kernel

The first test compares 42 combinations of length, dtype, and head dimension
against dense attention, including partial pages, GQA, online demotion, and
empty split partitions. The second compares full-model logits across stock
FP16, reconstructed-cache attention, and packed attention. A failure is saved
as a failure and the remaining quality experiment uses the dense reference.
Kernel compilation and these warm-ups are excluded from timed measurements.


In [ ]:
gates = {'requested_packed': REQUEST_PACKED_KERNEL, 'packed_status': 'not_requested'}
BACKEND = 'dense'
if REQUEST_PACKED_KERNEL:
    try:
        gates['attention_checks'] = kernel_gate()
        source = prefill_case(model, cases[0])
        gates['model_checks'] = model_gate(model, tokenizer, source, cases[0], settings)
        del source
        gates['packed_status'] = 'passed'
        BACKEND = 'packed'
    except Exception:
        gates['packed_status'] = 'failed'
        gates['error'] = traceback.format_exc()
        print(gates['error'])
        print('Continuing with dense reconstruction. This run cannot validate packed-kernel performance.')
save_json(OUTPUT / 'correctness-gates.json', gates)
print('Selected backend:', BACKEND, '| gate:', gates['packed_status'])
clean()


## Automatic selection and equal-budget evaluation

Recent, random, automatic, and evidence-oracle selectors preserve exactly the
same number of full FP16 pages. They also share one sink page and the rolling
recent window. All other eligible pages use INT4. Initial resident bytes,
including scales and pointer tables, must be identical or the experiment stops.
Residual INT4, residual INT8, stock FP16, and paged FP16 are additional baselines.

Automatic scores combine attention from the final prompt query with measured
K/V reconstruction error, averaged over the final four layers. Selection never
sees the gold answer or labeled evidence. Selector preparation time is reported
separately and included in its accounting. This selection is fixed per prompt;
age-based precision changes continue throughout decoding.

Every policy recomputes the final prompt token from the intervened prefix cache,
so the first generated answer token is independently evaluated. The gold answer
is used only for the separate teacher-forced diagnostic, including its EOS.


In [ ]:
raw_rows = []
diagnostics = []
selection_records = []
audits = []
for case in cases:
    print('\nStarting', case['case_id'], case['position'], 'tokens=', len(case['ids']), flush=True)
    sync()
    prefill_started = time.perf_counter()
    source = prefill_case(model, case)
    sync()
    prefill_seconds = time.perf_counter() - prefill_started
    prefix_length = len(case['ids']) - 1
    candidates = list(range(1, prefix_length // settings.block - settings.recent_pages))
    assert candidates, 'Increase context length to leave eligible archive pages.'
    count = max(1, int(len(candidates) * settings.keep_fraction))
    scores, reference_first, score_seconds = automatic_scores(model, source, case, candidates, settings)
    plans = {'native_fp16': frozenset(), 'paged_fp16': frozenset(),
             'int8': frozenset({0}), 'int4': frozenset({0})}
    for name in ('recent', 'random', 'automatic', 'oracle'):
        plans[name] = choose_pages(name, candidates, count, scores=scores,
                                   evidence=case['evidence_pages'], seed=settings.seed + len(selection_records)) | {0}
    selection_records.append({'case_id': case['case_id'], 'eligible_pages': candidates,
                              'extra_fp16_pages': count, 'automatic_scores': scores,
                              'selection_seconds': score_seconds,
                              'plans': {name: sorted(pages) for name, pages in plans.items()}})
    if RUN_SENSITIVITY_AUDIT:
        audits.append(sensitivity_audit(model, source, case, settings, scores, reference_first))
        save_json(OUTPUT / 'sensitivity-audit.json', audits)

    # Same gold token history, including EOS, for all policies.
    forced = tokenizer(case['answer'], add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
    _, reference = run_policy(model, tokenizer, source, case, settings,
                              'native_fp16', forced=forced, backend=BACKEND)
    initial_bytes = {}
    for policy, pages in plans.items():
        row, logits = run_policy(model, tokenizer, source, case, settings,
                                 policy, pages, backend=BACKEND, forced=forced)
        if policy in ('recent', 'random', 'automatic', 'oracle'):
            initial_bytes[policy] = row['initial_resident_bytes']
        divergence = kl_divergence(reference, logits)
        nll = -logits.log_softmax(-1).gather(-1, torch.tensor(forced).unsqueeze(-1)).mean()
        diagnostics.append({'case_id': case['case_id'], 'policy': policy,
                            'mean_kl': float(divergence.mean()), 'max_kl': float(divergence.max()),
                            'top1_agreement': float((reference.argmax(-1) == logits.argmax(-1)).float().mean()),
                            'gold_nll': float(nll), 'first_answer_kl': float(divergence[0]),
                            'initial_resident_bytes': row['initial_resident_bytes']})
        # Full free-generation warm-up, so the same allocation and decode paths are exercised.
        run_policy(model, tokenizer, source, case, settings, policy, pages, backend=BACKEND)
    assert len(set(initial_bytes.values())) == 1, f'Budget mismatch: {initial_bytes}'

    for repetition in range(settings.repeats):
        order = list(plans)
        random.Random(settings.seed + repetition + len(raw_rows)).shuffle(order)
        for policy in order:
            row, _ = run_policy(model, tokenizer, source, case, settings,
                                 policy, plans[policy], backend=BACKEND)
            selector_cost = score_seconds if policy == 'automatic' else 0.0
            row.update({'repeat': repetition, 'prompt_tokens': len(case['ids']),
                        'evidence_position': case['position'], 'prefill_seconds': prefill_seconds,
                        'selector_seconds': selector_cost,
                        'accounted_total_seconds': prefill_seconds + selector_cost + row['cache_plus_decode_seconds']})
            raw_rows.append(row)
        # Persist progress after each repetition, even if the session later disconnects.
        save_json(OUTPUT / 'raw-results.json', raw_rows)
        save_json(OUTPUT / 'teacher-forced.json', diagnostics)
        save_json(OUTPUT / 'selection.json', selection_records)
        print('Finished repetition', repetition + 1, '/', settings.repeats, flush=True)
    del source
    clean()


## Continuous-generation stress check

Retrieval answers are short, so they do not sufficiently exercise aging of
newly generated pages. This separate test forces 96 continuation tokens solely
to check cache lifecycle behavior. It is not an answer-accuracy measurement.
More archived pages must exist at the end than in the prefix-only cache.


In [ ]:
stress_case = cases[0]
stress_source = prefill_case(model, stress_case)
stress_settings = Settings(**asdict(settings))
stress_settings.max_new_tokens = 128
forced_stress = (tokenizer(' diagnostic continuation', add_special_tokens=False).input_ids * 96)[:96]
stress_row, _ = run_policy(model, tokenizer, stress_source, stress_case, stress_settings,
                           'int4', {0}, backend=BACKEND, forced=forced_stress)
initial_demotions_per_layer = max(0, (len(stress_case['ids']) - 1) // settings.block - settings.recent_pages - 1)
assert stress_row['demotions'] > initial_demotions_per_layer * model.config.num_hidden_layers
stress_row['scope'] = 'forced continuation lifecycle test; exclude from answer-accuracy statistics'
save_json(OUTPUT / 'continuous-cache-check.json', stress_row)
print('Continuous cache check passed:', stress_row['tokens'], 'tokens;', stress_row['demotions'], 'layer-page demotions')
del stress_source
clean()


## Per-case results, uncertainty, and report

Repetitions are timing replicates, not independent questions. Confidence
intervals resample cases after reducing repetitions. Six pilot questions give
weak generalization evidence; expand prompts/seeds before making a broad claim.
Prefill is measured once per case and amortized equally. Accounted total time
includes CPU snapshot/transfer, automatic selection where used, cache setup,
and decode. Separate measured cache-plus-decode times are also retained.
Prefill's peak GPU allocation is outside the decode-stage memory window.


In [ ]:
raw = pd.DataFrame(raw_rows)
quality = pd.DataFrame(diagnostics)
per_case = raw.groupby(['case_id', 'policy'], as_index=False).agg(
    answer_correct=('answer_correct', 'mean'), completed_correct=('completed_correct', 'mean'),
    peak_allocated_gib=('peak_allocated_gib', 'median'),
    peak_incremental_mib=('peak_incremental_mib', 'median'),
    end_incremental_mib=('end_incremental_mib', 'median'),
    resident_bytes=('resident_bytes', 'median'),
    cache_plus_decode_seconds=('cache_plus_decode_seconds', 'median'),
    accounted_total_seconds=('accounted_total_seconds', 'median'),
)
per_case = per_case.merge(quality, on=['case_id', 'policy'])
summary = per_case.groupby('policy', as_index=False).mean(numeric_only=True)
summary['resident_mib'] = summary.resident_bytes / 2**20
display(summary[['policy', 'completed_correct', 'resident_mib', 'mean_kl',
                 'peak_incremental_mib', 'accounted_total_seconds']].round(5))

paired = []
for control in ('recent', 'random', 'oracle', 'int4'):
    a = per_case[per_case.policy == 'automatic'].set_index('case_id')
    b = per_case[per_case.policy == control].set_index('case_id')
    for metric in ('mean_kl', 'completed_correct', 'accounted_total_seconds'):
        differences = (a[metric] - b[metric]).tolist()
        paired.append({'comparison': 'automatic minus ' + control, 'metric': metric,
                       'mean_difference': statistics.mean(differences),
                       'case_bootstrap_95pct': bootstrap_interval(differences, settings.seed)})
save_json(OUTPUT / 'paired-comparisons.json', paired)
raw.to_csv(OUTPUT / 'raw-results.csv', index=False)
per_case.to_csv(OUTPUT / 'per-case.csv', index=False)
summary.to_csv(OUTPUT / 'summary.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(summary.policy, summary.resident_mib)
axes[0].set_ylabel('End-of-run KV payload + metadata (MiB)')
axes[1].bar(summary.policy, summary.completed_correct)
axes[1].set_ylabel('Completed exact-answer fraction')
axes[1].set_ylim(0, 1.05)
axes[2].scatter(summary.resident_mib, summary.mean_kl)
for row in summary.itertuples():
    axes[2].annotate(row.policy, (row.resident_mib, row.mean_kl), fontsize=8)
axes[2].set_xlabel('Resident cache (MiB)')
axes[2].set_ylabel('Mean teacher-forced KL vs stock FP16')
for ax in axes[:2]:
    ax.tick_params(axis='x', rotation=65)
fig.tight_layout()
fig.savefig(OUTPUT / 'summary.png', dpi=160, bbox_inches='tight')
plt.show()

report = [
    '# AnchorKV T4 experiment report', '',
    f'GPU: {environment["gpu"]}; model: {settings.model_id}@{settings.revision}.',
    f'Backend: {BACKEND}; packed correctness gate: {gates["packed_status"]}.',
    f'Independent cases: {len(cases)}; timing repetitions: {settings.repeats}.', '',
    '## Results', '', '```', summary.to_string(index=False), '```', '',
    '## Interpretation boundaries', '',
    '- Equal budgets apply to recent/random/automatic/oracle initial prefix caches.',
    '- INT4 and INT8 baselines keep the same FP16 sink and rolling recent window.',
    '- Automatic scores use only prompt-query attention and KV quantization error.',
    '- Oracle selection uses labeled evidence; it is not an automatic result.',
    '- Cache allocation/packing and CPU transfer are included in cache-plus-decode time.',
    '- Automatic selection and shared prefill are included in accounted total time.',
    '- Decode-stage peak memory excludes prefill; payload bytes include scales and tables.',
    '- Final cache sizes may differ because policies can generate different token counts.',
    '- The pilot is too small for broad accuracy claims. Repeated timings are not extra cases.',
    '- Production batching, serving integration, and model-generated directive control remain future work.',
]
(OUTPUT / 'report.md').write_text('\n'.join(report), encoding='utf-8')
print('Report:', OUTPUT / 'report.md')


## Download the complete evidence bundle

Return this zip for analysis. It includes raw generations, exact prompts,
settings, package/model/source versions, selector choices, numerical gates,
lifecycle tests, per-case tables, confidence intervals, and the report.
Passing the notebook establishes results for this configuration. Remaining
research is replication across larger prompt sets, seeds, models, and longer
reasoning tasks. Production use additionally needs batching, an allocator and
serving-engine integration, and performance tuning against strong baselines.


In [ ]:
source_dir = OUTPUT / 'runtime-source'
source_dir.mkdir(exist_ok=True)
for filename, source in EMBEDDED_SOURCES.items():
    (source_dir / filename).write_text(source, encoding='utf-8')
archive = shutil.make_archive(str(OUTPUT), 'zip', OUTPUT)
print('Results:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Download the zip from the path above.')
